# 74 — End-to-end: SASRec recall + wRRF union -> LGBM rerank -> nDCG, + Blind-A submission

One notebook for the whole improved pipeline.

Stage 1 (recall): content-fused SASRec as a 4th channel in wrrf_union_v1. Reports recall@{20,100} for union vs union+SASRec (the G1 result; +0.0586 @100 on full dev).

Stage 2 (rerank train): build LGBM LambdaRank features WITH the SASRec channel (sasrec_rank_inv feature) and train two models — with and without that feature — reporting held-out val nDCG@20 (the G2 number + its control).

Stage 3 (end-to-end DEV nDCG): union+SASRec recall@100 -> LGBM rerank -> nDCG@20 on dev (we have golds here). Compares recall-only vs LGBM(no sasrec feat) vs LGBM(+sasrec feat).

Stage 4 (Blind-A submission, separate logic at the end): run the full pipeline (config 191 = union+SASRec -> LGBM(+sasrec) -> v5-kto responder) over the 80 Blind-A queries and package prediction.json. NOTE: Blind-A has no public golds — nDCG there is scored by the CodaBench server, not locally. Stage 3 is the local nDCG signal.

Run order: cells top to bottom. Stages 1-3 are fast; Stage 4 is the long responder run (~35-65 min) — run it only when Stage 3 confirms the lift.

In [ ]:
# 1) Setup. Disable JAX GPU preallocation BEFORE any import pulls JAX in
# (datasets/transformers import JAX transitively; it grabs ~75% VRAM on first use).
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'recall-union-lgbm'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

# Symlink the persistent caches from Drive. retrieval_v2 holds sasrec/, lgbm/,
# ctx_cache/; dense holds the Qwen query/catalog cache for dense_metadata_qwen3.
DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('retrieval_v2', 'recsys2026_retrieval_v2_cache'),
    ('dense', 'recsys2026_dense_cache'),
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

# Deps: retrieval stack + lightgbm (Stages 1-3) + responder/inference (Stage 4).
!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11'     'datasets' 'pandas<3.0' 'tqdm' 'huggingface_hub' 'sentence-transformers>=3.0'     'FlagEmbedding>=1.3' 'bm25s' 'lightgbm' 'scikit-learn'     'omegaconf' 'pyyaml' 'trl>=0.12.0' 'torchao>=0.17'


## Stage 1 — recall: content-fused SASRec as the 4th union channel

In [ ]:
# 3) Ensure the content-fused SASRec checkpoint exists (sasrec_v1). Trains it
# only if missing (it persists on the Drive cache across runtimes).
import os, sys
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
CACHE_DIR = '/content/recsys2026/experiments/cache'
ITEM_DB = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
CORPUS = ['track_name', 'artist_name', 'album_name']
SASREC_CKPT = f'{CACHE_DIR}/retrieval_v2/sasrec/sasrec_v1/sasrec.pt'
if os.path.exists(SASREC_CKPT):
    print('[sasrec] checkpoint present, skipping train:', SASREC_CKPT)
else:
    print('[sasrec] training content-fused SASRec (sasrec_v1, ~10 epochs)...')
    !cd /content/recsys2026 && python -u scripts/train_sasrec.py \
        --cache-dir {CACHE_DIR} --out sasrec_v1 --epochs 10


In [ ]:
# 4) Build the FULL dev eval set + report recall@{20,100} for union vs union+SASRec.
# A2 FIX (2026-05-30): query now appends listener_goal, matching the production
# crs_baseline query. Blind-A nDCG 0.29 >> old dev 0.16 was largely this mismatch;
# the dev harness was pessimistic. Downstream stages reuse this query.
# Defines the shared dev variables reused by Stage 3.
import numpy as np
import pandas as pd
from datasets import load_dataset
from mcrs.db_item.music_catalog import MusicCatalogDB
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.sasrec_model import build_user_dialog

item_db = MusicCatalogDB(ITEM_DB, ['all_tracks'], CORPUS)
dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')

queries, golds, user_ids, played, user_dialogs = [], [], [], [], []
goal_categories, goal_specificities, turn_numbers = [], [], []
for sess in dev:
    df = pd.DataFrame(sess['conversations'])
    goal = sess.get('conversation_goal') or {}
    goal_txt = (goal.get('listener_goal') or '').strip()  # A2: prod query includes this
    for _, music in df[df['role'] == 'music'].iterrows():
        tn = int(music['turn_number'])
        prior = df[(df['turn_number'] < tn) |
                   ((df['turn_number'] == tn) & (df['role'] == 'user'))]
        lines = []
        for _, t in prior.iterrows():
            role = 'assistant' if t['role'] == 'music' else t['role']
            content = item_db.id_to_metadata(t['content']) if t['role'] == 'music' else t['content']
            lines.append(f'{role}: {content}')
        _q = chr(10).join(lines)
        if goal_txt:
            _q = _q + chr(10) + 'goal: ' + goal_txt  # A2: train/serve parity w/ crs_baseline
        queries.append(_q)
        user_dialogs.append(build_user_dialog(prior.to_dict('records')))
        golds.append(music['content'])
        user_ids.append(sess.get('user_id'))
        played.append(list(df[(df['role'] == 'music') & (df['turn_number'] < tn)]['content']))
        goal_categories.append(goal.get('category'))
        goal_specificities.append(goal.get('specificity'))
        turn_numbers.append(tn)
ctx = [{'history_tids': p, 'user_dialog': ud} for p, ud in zip(played, user_dialogs)]
print('[dev] built', len(queries), 'turns')

def recall_at(cands, k):
    return float(np.mean([1.0 if g in c[:k] else 0.0 for c, g in zip(cands, golds)]))

base = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                             CACHE_DIR, extra_config={})
sas = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                            CACHE_DIR, extra_config={'use_sasrec': True, 'w_sasrec': 1.0})
cb = base.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
cs = sas.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
print('=== Stage 1 recall (FULL dev, n=' + str(len(golds)) + ') ===')
print('  union (3-chan) : @20=' + str(round(recall_at(cb, 20), 4)) + ' @100=' + str(round(recall_at(cb, 100), 4)))
print('  union + SASRec : @20=' + str(round(recall_at(cs, 20), 4)) + ' @100=' + str(round(recall_at(cs, 100), 4)))
print('  delta @100     :', round(recall_at(cs, 100) - recall_at(cb, 100), 4))


## Stage 2 — LGBM rerank: build features (with SASRec) + train (with vs without the SASRec feature)

In [ ]:
# 6) Build LGBM LambdaRank features from train sessions, WITH the SASRec channel
# (emits the real sasrec_rank_inv column). Two seeds -> train + val splits.
import os
LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
os.makedirs(LGBM_DIR, exist_ok=True)
TRAIN_FEAT = f'{LGBM_DIR}/lgbm_train_sasrec.parquet'
VAL_FEAT   = f'{LGBM_DIR}/lgbm_val_sasrec.parquet'
if os.path.exists(TRAIN_FEAT) and os.path.exists(VAL_FEAT):
    print('[lgbm] feature parquets present, skipping build')
else:
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 2000 --topk 100 --seed 42 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {TRAIN_FEAT}
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 400 --topk 100 --seed 7 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {VAL_FEAT}


In [ ]:
# 7) Train THREE LGBM models on the SAME feature build to isolate the in-sample
# model-derived feature leak (see project_sasrec_lgbm_feature_leak memory):
#   lgbm_sasrec_v1   : all columns (sasrec_rank_inv + cfbpr_score present)
#   lgbm_nosasrec_v1 : sasrec_rank_inv dropped (cfbpr_score still present)
#   lgbm_clean_v1    : BOTH sasrec_rank_inv AND cfbpr_score dropped  <-- LEAK TEST
# train_lgbm_ranker auto-selects every non-id column as a feature, so dropping a
# column is the clean one-axis control. Cheap (CPU, no-GPU) confirmatory test:
# if lgbm_clean_v1 matches/beats recall-only on dev (cell 9) while the others lose,
# the leak is confirmed as the reranker's whole problem.
import os, json
import pandas as pd
LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
TRAIN_FEAT = f'{LGBM_DIR}/lgbm_train_sasrec.parquet'
VAL_FEAT   = f'{LGBM_DIR}/lgbm_val_sasrec.parquet'
TRAIN_NS   = f'{LGBM_DIR}/lgbm_train_nosasrec.parquet'
VAL_NS     = f'{LGBM_DIR}/lgbm_val_nosasrec.parquet'
TRAIN_CL   = f'{LGBM_DIR}/lgbm_train_clean.parquet'
VAL_CL     = f'{LGBM_DIR}/lgbm_val_clean.parquet'
# nosasrec control: drop only sasrec_rank_inv
for src, dst in [(TRAIN_FEAT, TRAIN_NS), (VAL_FEAT, VAL_NS)]:
    df = pd.read_parquet(src)
    df.drop(columns=[c for c in ['sasrec_rank_inv'] if c in df.columns]).to_parquet(dst, index=False)
print('[lgbm] built no-sasrec control parquets')
# clean (leak test): drop BOTH model-derived leaked features
LEAKED = ['sasrec_rank_inv', 'cfbpr_score']
for src, dst in [(TRAIN_FEAT, TRAIN_CL), (VAL_FEAT, VAL_CL)]:
    df = pd.read_parquet(src)
    df.drop(columns=[c for c in LEAKED if c in df.columns]).to_parquet(dst, index=False)
print('[lgbm] built clean (no-leak) parquets \u2014 dropped', LEAKED)

!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRAIN_FEAT} --val-features {VAL_FEAT} \
    --output-dir {LGBM_DIR}/lgbm_sasrec_v1
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRAIN_NS} --val-features {VAL_NS} \
    --output-dir {LGBM_DIR}/lgbm_nosasrec_v1
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRAIN_CL} --val-features {VAL_CL} \
    --output-dir {LGBM_DIR}/lgbm_clean_v1

print('\n=== Stage 2 held-out val nDCG@20 (LGBM internal \u2014 leak-inflated, see cell 9 for honest dev) ===')
for name in ['lgbm_clean_v1', 'lgbm_nosasrec_v1', 'lgbm_sasrec_v1']:
    meta = json.load(open(f'{LGBM_DIR}/{name}/metadata.json'))
    print('  ' + name + ': val_ndcg@20=' + str(round(meta['best_val_ndcg20'], 4)) +
        '  (' + str(len(meta['features'])) + ' features)')


## Stage 3 — end-to-end DEV nDCG@20: recall -> rerank

In [ ]:
# 9) End-to-end on dev: union+SASRec recall@100 -> LGBM rerank top-20 -> nDCG@20.
# All LGBM models rerank the SAME union+SASRec pool, so this isolates the rerank
# feature set. The SASRec per-candidate rank is fed exactly as crs_baseline does in
# prod; LGBM_RERANKER only uses features listed in its own metadata, so the clean
# model harmlessly ignores efpc's sasrec_rank and skips cfbpr_score.
# LEAK TEST READING: recall-only is the bar (0.1473 in the run that found the leak).
#   - lgbm_clean_v1 (no leaked feats) >= recall-only  => leak WAS the problem; this
#     leak-free reranker is shippable with NO GPU. OOF only needed to ADD sasrec
#     value on top.
#   - lgbm_clean_v1 still < recall-only               => leak isn't the whole story;
#     OOF would not have helped alone. Investigate label sparsity / wrrf_rank
#     contamination / train-set size next.
import math
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
fused100 = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 100)
sidx = labels.index('sasrec_seq')
efpc = []
for qi, cands in enumerate(fused100):
    rm = {tid: r + 1 for r, tid in enumerate(per_sub[sidx][qi])}
    efpc.append([{'sasrec_rank': rm.get(tid, 10000)} for tid in cands])
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2)
                break
    return s / len(golds)

print('=== Stage 3 end-to-end DEV (n=' + str(len(golds)) + ') ===')
print('  recall@100 pool ceiling      :', round(recall_at(fused100, 100), 4))
recall_only = ndcg20([r[:20] for r in fused100])
print('  nDCG@20 recall-only (no rerank):', round(recall_only, 4), '  <-- bar to beat')
for name, sub in [('LGBM clean (no leaked feats)', 'lgbm_clean_v1'),
                  ('LGBM (no sasrec feat)      ', 'lgbm_nosasrec_v1'),
                  ('LGBM (+ sasrec feat)       ', 'lgbm_sasrec_v1')]:
    rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                       model_path=f'{CACHE_DIR}/retrieval_v2/lgbm/{sub}')
    reranked = rr.rerank(queries, fused100, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories,
                         goal_specificities=goal_specificities,
                         user_profiles_raw=[None] * len(queries),
                         extra_features_per_candidate=efpc, extra_session_info=esi)
    score = ndcg20(reranked)
    flag = '  BEATS recall-only' if score >= recall_only else ''
    print('  nDCG@20 ' + name + ' :', round(score, 4), flag)


## Stage 4 — Blind-A submission (separate logic)

Runs the full pipeline via config 194 (union+SASRec -> lgbm_clean_full (leak-free) -> v5-kto responder) over the 80 Blind-A queries and packages prediction.json for CodaBench. This is the LONG cell (~35-65 min). Blind-A nDCG is server-scored — there are no local golds; use Stage 3 as the local signal before submitting.

Requires Stage 11 to have written lgbm_clean_full to the Drive cache (config 194 points at it).

In [ ]:
# 11) Blind-A inference -> prediction.json -> zip for CodaBench.
# Ships config 194 = union+SASRec -> lgbm_clean_full (leak-free, 15k-session, the
# nb74 Stage 11 best, dev nDCG@20 0.1623) -> v5-kto responder. NOT config 191
# (leaked reranker lgbm_sasrec_v1, dev 0.1194 — do not ship).
import os
TID = '194-union-sasrec-lgbm-cleanfull-v5kto-blindA'
PRED_PATH = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/{TID}.json'
# config 194 reranker_model_path must resolve to lgbm_clean_full (built in Stage 11).
assert os.path.exists(f'{CACHE_DIR}/retrieval_v2/lgbm/lgbm_clean_full/booster.txt'), \
    'Run Stage 11 first — lgbm_clean_full booster missing.'

%cd /content/recsys2026/music-crs-baselines
!python run_inference_blindset.py --tid {TID} --batch_size 8 2>&1 | tail -60
%cd /content/recsys2026

import json
preds = json.load(open(PRED_PATH))
n = len(preds) if isinstance(preds, list) else len(preds)
print('[blindA] prediction entries:', n)
assert n == 80, f'expected 80 Blind-A entries, got {n} — DO NOT submit'
print('[blindA] sample keys:', list(preds[0].keys()))

# Optional strict precheck (catalog membership + schema):
#   !python scripts/precheck_prediction.py {PRED_PATH}

import zipfile, datetime
zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{datetime.date.today().isoformat()}-{TID}.zip'
os.makedirs(os.path.dirname(zip_path), exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_PATH, arcname='prediction.json')   # MUST be at zip root for CodaBench
print('[blindA] submission zip ready:', zip_path)
print('[blindA] upload to https://www.codabench.org/competitions/ and append scores via scripts/blind_a_score_tracker.py')


## Stage 5 - OOF leak-free test: does cross-fit sasrec_rank_inv beat the clean reranker?

The clean reranker (0.1558) drops sasrec_rank_inv entirely. This stage rebuilds
that feature LEAK-FREE via K-fold out-of-fold cross-fitting (each fold scored by
a SASRec that never trained on it), retrains the LGBM, and compares on dev:
- if lgbm_oof_v1 > 0.1558  -> leak-free SASRec rank ADDS value; keep it
- if lgbm_oof_v1 <= 0.1558 -> SASRec rank is dead even when honest; drop it
COST: K SASRec retrains (~25-40 min each on Blackwell) + 2K feature builds.

In [ ]:
# OOF (out-of-fold) cross-fit of sasrec_rank_inv, then retrain LGBM + compare.
# Requires cell 1 (CACHE_DIR/ITEM_DB/CORPUS) and cell 4 (sas, queries, user_ids,
# ctx, played, golds, turn_numbers, goal_categories, goal_specificities).
# Idempotent: each heavy step skips if its artifact exists, so re-running after
# an interruption does not redo finished folds.
import os, math
import pandas as pd
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

K = 5  # OOF folds = K SASRec retrains. Lower K is cheaper but weaker per-fold model.
LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
SAS_DIR  = f'{CACHE_DIR}/retrieval_v2/sasrec'

# 1) Train K SASRec models, each HOLDING OUT fold k from training.
for k in range(K):
    out = f'sasrec_oof_{K}_{k}'
    if os.path.exists(f'{SAS_DIR}/{out}/sasrec.pt'):
        print(f'[oof] SASRec {out} present, skip'); continue
    !cd /content/recsys2026 && python -u scripts/train_sasrec.py \
        --cache-dir {CACHE_DIR} --out {out} \
        --d 256 --max-seq 50 --epochs 5 --batch-size 256 --lr 1e-3 \
        --oof-fold {k} --oof-num-folds {K}

# 2) Build leak-free feature parquets fold by fold (fold k scored by
#    sasrec_oof_K_k, which never saw fold k). Same seed/n-sessions as cell 6.
train_parts, val_parts = [], []
for k in range(K):
    tp = f'{LGBM_DIR}/oof_train_{K}_{k}.parquet'
    vp = f'{LGBM_DIR}/oof_val_{K}_{k}.parquet'
    if not os.path.exists(tp):
        !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
            --n-sessions 2000 --topk 100 --seed 42 \
            --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_oof_{K}_{k} \
            --oof-fold {k} --oof-num-folds {K} \
            --cache-dir {CACHE_DIR} --out {tp}
    if not os.path.exists(vp):
        !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
            --n-sessions 400 --topk 100 --seed 7 \
            --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_oof_{K}_{k} \
            --oof-fold {k} --oof-num-folds {K} \
            --cache-dir {CACHE_DIR} --out {vp}
    train_parts.append(tp); val_parts.append(vp)

# 3) Concat per-fold parquets, drop cfbpr_score (permanently leaked, not OOF-able).
TRAIN_OOF = f'{LGBM_DIR}/lgbm_train_oof.parquet'
VAL_OOF   = f'{LGBM_DIR}/lgbm_val_oof.parquet'
tr = pd.concat([pd.read_parquet(x) for x in train_parts], ignore_index=True)
va = pd.concat([pd.read_parquet(x) for x in val_parts], ignore_index=True)
assert 'sasrec_rank_inv' in tr.columns, 'OOF train parquet missing sasrec_rank_inv'
tr.drop(columns=[c for c in ['cfbpr_score'] if c in tr.columns]).to_parquet(TRAIN_OOF, index=False)
va.drop(columns=[c for c in ['cfbpr_score'] if c in va.columns]).to_parquet(VAL_OOF, index=False)
print(f'[oof] concatenated {K} folds -> train {len(tr)} rows, val {len(va)} rows')

# 4) Retrain LGBM on the OOF features (sasrec_rank_inv now honest).
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRAIN_OOF} --val-features {VAL_OOF} \
    --output-dir {LGBM_DIR}/lgbm_oof_v1

# 5) Compare on dev. dev is the test split -> SASRec never trained on it, so the
#    dev sasrec rank fed via efpc is already honest for every model.
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
fused100 = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 100)
sidx = labels.index('sasrec_seq')
efpc = []
for qi, cands in enumerate(fused100):
    rm = {tid: r + 1 for r, tid in enumerate(per_sub[sidx][qi])}
    efpc.append([{'sasrec_rank': rm.get(tid, 10000)} for tid in cands])
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

print('\n=== Stage 5 OOF comparison DEV (n=' + str(len(golds)) + ') ===')
recall_only = ndcg20([r[:20] for r in fused100])
print('  recall-only (no rerank)      :', round(recall_only, 4))
scores = {}
for name, sub in [('LGBM clean_v1  (2k, control) ', 'lgbm_clean_v1'),
                  ('LGBM clean_full(15k, shipped)', 'lgbm_clean_full'),
                  ('LGBM OOF       (2k, +sasrec) ', 'lgbm_oof_v1')]:
    mp = f'{LGBM_DIR}/{sub}'
    if not os.path.exists(f'{mp}/booster.txt'):
        print('  ' + name + ' : (missing, skipped)'); continue
    rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, model_path=mp)
    reranked = rr.rerank(queries, fused100, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories,
                         goal_specificities=goal_specificities,
                         user_profiles_raw=[None] * len(queries),
                         extra_features_per_candidate=efpc, extra_session_info=esi)
    sc = ndcg20(reranked)
    scores[sub] = ndcg20(reranked)
    print('  ' + name + ' :', round(scores[sub], 4))
oof = scores.get('lgbm_oof_v1'); cv1 = scores.get('lgbm_clean_v1'); cf = scores.get('lgbm_clean_full')
print('\n-- DECISION (OOF at 2k) --')
if oof is not None and cv1 is not None:
    d = oof - cv1
    print('  feature test : OOF-2k %+.4f vs clean-2k (%s)' % (d, 'HELPS' if d > 0.002 else 'inert/redundant'))
if oof is not None and cf is not None:
    print('  ship test    : OOF-2k %+.4f vs clean-full 0.1623 shipped' % (oof - cf))
    print('  -> rebuild OOF at 15k ONLY if 2k feature test HELPS and reaches clean-full;')
    print('     else honest sasrec rank is redundant with wrrf_rank -> drop, keep clean_full.')


## Stage 6 - recall ceiling diagnostic: WHERE do we lose nDCG?

nDCG@20 is capped by recall@100 (0.4961). The reranker only reorders what recall
surfaced, so to climb the leaderboard we must raise recall, not rerank harder.
This cell (no GPU, reuses cell-4 data) breaks the ceiling down:
- per-channel miss rate (which channel finds what)
- ALL-channel miss = the hard wall no reranker can cross
- new-artist share of misses (cold-start / content gap)
- in-pool-but-past-100 (cheap recall@larger-K headroom)
The numbers pick the next lever: content modality vs wider pool vs channel swap.

In [ ]:
# Recall ceiling diagnostic. Requires cell 4 (sas, queries, user_ids, ctx,
# golds, played, item_db, recall_at, cs).
from mcrs.retrieval_modules.rrf import RRF_MODEL

per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
N = len(golds)
print(f'=== Stage 6 recall ceiling diagnostic (dev n={N}) ===')
print(f'channels: {labels}')

# 1) per-channel recall@100 (did this channel surface the gold at all?)
print('\n-- per-channel recall@100 (gold present in that channel top-100) --')
in_chan = [[False] * N for _ in labels]
for s in range(len(labels)):
    for q in range(N):
        in_chan[s][q] = golds[q] in set(per_sub[s][q])
    print(f'  {labels[s]:24s}: {sum(in_chan[s]) / N:.4f}')

# 2) union ceiling (any channel has it) — should track recall@100 of the fusion
any_chan = [any(in_chan[s][q] for s in range(len(labels))) for q in range(N)]
union_ceiling = sum(any_chan) / N
print(f'\n  UNION ceiling (any channel)      : {union_ceiling:.4f}')
print(f'  fused recall@100 (cs, post-RRF)  : {recall_at(cs, 100):.4f}')

# 3) the hard wall: golds NO channel surfaced
miss_idx = [q for q in range(N) if not any_chan[q]]
print(f'\n  ALL-CHANNEL MISS (hard wall)     : {len(miss_idx)}/{N} = {len(miss_idx)/N:.4f}')
print('  ^ no reranker can ever recover these; only a NEW recall signal can.')

# 4) new-artist analysis: of the all-channel misses, how many have an artist
#    that never appeared in the session history (cold-start / content gap)?
def artist_of(tid):
    m = item_db.metadata_dict.get(tid, {})
    a = m.get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

new_artist_miss = 0
known_artist_miss = 0
for q in miss_idx:
    g_art = artist_of(golds[q])
    hist_arts = {artist_of(t) for t in played[q]}
    if g_art is not None and g_art in hist_arts:
        known_artist_miss += 1
    else:
        new_artist_miss += 1
if miss_idx:
    print(f'\n  of the {len(miss_idx)} hard-wall misses:')
    print(f'    new-artist (not in session history): {new_artist_miss} '
          f'({new_artist_miss/len(miss_idx):.1%})')
    print(f'    known-artist (in history, still missed): {known_artist_miss} '
          f'({known_artist_miss/len(miss_idx):.1%})')

# 5) cheap headroom: golds in the fused pool but ranked past 100 would need a
#    wider pool. Compare recall at 100 vs 200/500 of the fusion.
weights = [s['weight'] for s in sas.subs]
for wider in (200, 500):
    fused_wide = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, wider)
    print(f'  fused recall@{wider:<4d}              : {recall_at(fused_wide, wider):.4f}')
print('  ^ gain from 100 to 200/500 = cheap recall headroom (just widen topk).')

print('\n-- READING --')
print('  high new-artist share  -> add a content channel (lyrics modality) / stronger content')
print('  big jump at @200/@500  -> widen the candidate pool (nearly free)')
print('  one channel dominates  -> rebalance or replace a dead channel')


## Stage 7 - fix the content channel: dense_metadata_qwen3 A/B (no GPU train)

Stage 6 found dense_metadata_qwen3 at recall@100=0.0894 (near-dead) while the
new-artist hard wall is 98.8% of misses -- the content channel is exactly what
should catch those, and it's broken. Two suspected causes:
  P1 instruct prefix: the union uses dense_metadata_qwen3 (instruct=None), but
     the encoder Qwen3-Embedding-0.6B is ASYMMETRIC and needs a query instruct
     prefix. dense_metadata_qwen3_instruct applies it.
  P2 query pollution: cell 4's query is id_to_metadata text -> 'track_id: <uuid>,
     ...' for every prior music turn. UUIDs are noise to an embedder (bm25
     tolerates it; dense averages it in). user_dialog (user-turns only) is clean.
This A/B isolates each on dev recall@100. Track embeddings are precomputed, so
only the QUERY side re-encodes (minutes, no training). The winner is a one-line
union spec change (swap to _instruct and/or feed the clean query).

In [ ]:
# Dense channel A/B. Requires cell 4 (queries, user_ids, golds, recall_at,
# user_dialogs, ITEM_DB, CORPUS, CACHE_DIR).
from mcrs.retrieval_modules import load_retrieval_module

# Clean query = user-turns-only dialog (no uuid/metadata pollution). cell 4 built
# user_dialogs alongside queries; fall back to queries if absent.
clean_q = user_dialogs if 'user_dialogs' in dir() else queries

def dense_recall(retr_type, q_list, label):
    d = load_retrieval_module(retr_type, ITEM_DB, ['all_tracks'], CORPUS,
                              CACHE_DIR, extra_config={})
    cand = d.batch_text_to_item_retrieval(q_list, topk=100, user_ids=user_ids)
    r20 = recall_at(cand, 20)
    r100 = recall_at(cand, 100)
    print(f'  {label:38s}: @20={r20:.4f} @100={r100:.4f}')
    return r100

print('=== Stage 7 dense channel A/B (dev recall@100) ===')
print('  baseline reference: dense in union today = 0.0894 @100\n')
a = dense_recall('dense_metadata_qwen3',          queries, 'A raw + polluted query (reproduce)')
b = dense_recall('dense_metadata_qwen3_instruct', queries, 'B + instruct prefix (P1)')
c = dense_recall('dense_metadata_qwen3_instruct', clean_q, 'C instruct + clean user-dialog (P1+P2)')

best = max([('A', a), ('B', b), ('C', c)], key=lambda kv: kv[1])
print(f'\n  winner: {best[0]} @100={best[1]:.4f}  (vs current 0.0894)')
print('  A->B gain = instruct-prefix effect; B->C gain = clean-query effect.')
print('  Apply the winner to the union: _wrrf_union_v1_specs dense sub ->')
print('  dense_metadata_qwen3_instruct' + (' + feed clean query' if best[0] == 'C' else ''))


## Stage 8 - does the dense fix lift the UNION? (the decision gate)

Stage 7 doubled the dense channel in isolation (0.0894 -> 0.1789 instruct). But
the union recall@100 (0.4961) only rises if dense's now-working hits include
golds the OTHER channels miss -- especially new-artist golds (98.8% of the hard
wall). This cell measures that directly:
  - union recall@100 with dense FIXED (instruct) vs the 0.4961 baseline
  - of the OLD hard-wall misses, how many the fixed dense now rescues
  - how many of those rescues are new-artist (the wall we care about)
This is the gate: a real union lift + new-artist rescues = dense is the lever and
a STRONGER embedder (Qwen3-Embedding-4B on the A100-40GB) will compound. A flat
union = dense hits are redundant with bm25/sasrec; don't spend GPU on 4B.

In [ ]:
# Stage 8: union dense-fix recall gate. Requires cell 4 (queries, user_ids, ctx,
# golds, played, item_db, recall_at) and the dense-instruct fix in the factory.
from mcrs.retrieval_modules import load_retrieval_module

# Rebuild the union TWO ways: dense_instruct on (new default) vs off (old raw).
sas_fixed = load_retrieval_module(
    'wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
    extra_config={'use_sasrec': True, 'w_sasrec': 1.0})  # dense_instruct defaults True
sas_raw = load_retrieval_module(
    'wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
    extra_config={'use_sasrec': True, 'w_sasrec': 1.0, 'dense_instruct': False})

cs_fixed = sas_fixed.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
cs_raw   = sas_raw.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)

N = len(golds)
r_raw  = recall_at(cs_raw, 100)
r_fix  = recall_at(cs_fixed, 100)
print('=== Stage 8 union recall@100: dense fix gate (dev n=%d) ===' % N)
print('  union (dense RAW, old)     : %.4f' % r_raw)
print('  union (dense INSTRUCT, new): %.4f' % r_fix)
print('  delta                      : %+.4f' % (r_fix - r_raw))

def artist_of(tid):
    m = item_db.metadata_dict.get(tid, {})
    a = m.get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

# Of golds the OLD union missed @100, how many does the FIXED union now rescue,
# and how many of those are new-artist (not in session history)?
old_miss = [q for q in range(N) if golds[q] not in set(cs_raw[q][:100])]
rescued = [q for q in old_miss if golds[q] in set(cs_fixed[q][:100])]
rescued_new_artist = 0
for q in rescued:
    g = artist_of(golds[q]); hist = {artist_of(t) for t in played[q]}
    if not (g is not None and g in hist):
        rescued_new_artist += 1
print('\n  old-union misses @100        : %d' % len(old_miss))
print('  rescued by dense fix         : %d' % len(rescued))
if rescued:
    print('    of which new-artist        : %d (%.1f%%)' %
          (rescued_new_artist, 100.0 * rescued_new_artist / len(rescued)))

print('\n-- DECISION --')
if r_fix - r_raw >= 0.005:
    print('  union LIFTED by the free fix -> dense is a live lever.')
    print('  -> a stronger embedder (Qwen3-Embedding-4B, A100-40GB) should compound.')
else:
    print('  union ~flat -> dense hits are largely redundant with bm25/sasrec.')
    print('  -> 4B upgrade likely low ROI; look elsewhere for new-artist recall.')


## Stage 9 - wider pool: convert latent recall into nDCG (GAP 1)

Stage 6 showed fused recall@100=0.4961 but @500=0.5675 -- +0.0714 of golds are
ALREADY surfaced at rank 100-500, just truncated by the top-100 pool. The reranker
(lgbm_clean_v1) reorders whatever pool it's given, so feeding it 200/500 candidates
lets it pull those deeper golds into the top-20 -- IF its features rank them well.
This cell reranks the SAME union+SASRec rankings at pool sizes {100,200,500} and
reports dev nDCG@20. Free on retrieval (re-fuses cached per-sub rankings).
  rising nDCG with pool size -> latent recall converts; widen the prod pool.
  flat/falling -> reranker can't surface deep golds; pool width won't help nDCG.

In [ ]:
# Stage 9: wider-pool nDCG. Requires cell 4 (sas, queries, user_ids, ctx, golds,
# played, turn_numbers, goal_categories, goal_specificities) + lgbm_clean_v1.
import math
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

# Per-sub rankings once (cached); re-fuse at each pool size. sas already has the
# instruct dense fix (cell 4 default).
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
sidx = labels.index('sasrec_seq')

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

def recall_at_pool(fused, k):
    return sum(1.0 for f, g in zip(fused, golds) if g in set(f[:k])) / len(golds)

rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                   model_path=f'{CACHE_DIR}/retrieval_v2/lgbm/lgbm_clean_v1')
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

print('=== Stage 9 wider-pool nDCG@20 (dev n=%d, lgbm_clean_v1) ===' % len(golds))
print('  recall-only top-20 of fused@100 baseline = ndcg ref\n')
print('  %-6s %-12s %-12s' % ('pool', 'recall@pool', 'nDCG@20'))
for POOL in (100, 200, 500):
    fused = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, POOL)
    efpc = []
    for qi, cands in enumerate(fused):
        rm = {tid: r + 1 for r, tid in enumerate(per_sub[sidx][qi])}
        efpc.append([{'sasrec_rank': rm.get(tid, 10000)} for tid in cands])
    reranked = rr.rerank(queries, fused, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories,
                         goal_specificities=goal_specificities,
                         user_profiles_raw=[None] * len(queries),
                         extra_features_per_candidate=efpc, extra_session_info=esi)
    print('  %-6d %-12.4f %-12.4f' % (POOL, recall_at_pool(fused, POOL), ndcg20(reranked)))

print('\n-- READING --')
print('  nDCG rises with pool -> latent recall converts; set prod retrieval_topk higher.')
print('  nDCG flat/falls      -> reranker cannot surface deep golds; pool width is not the lever.')


## Stage 10 - reranker discrimination diagnostic: WHY can't it rank golds?

Stage 9 flipped the diagnosis: recall rose +0.0726 (pool 100->500) but nDCG@20
stayed flat 0.1558. The reranker can't surface deep golds. This cell measures WHY
(no GPU): lgbm_clean_v1 feature gains, and for golds PRESENT in pool@100 their rank
AFTER rerank + raw->reranked movement + net top-20 capture. If present-golds land
at high median rank, no feature separates gold from noise -> need data/features.

In [ ]:
# Stage 10: reranker discrimination diagnostic. Requires cell 4 (sas, queries,
# user_ids, ctx, golds, played, turn_numbers, goal_categories, goal_specificities).
import json, numpy as np
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
meta = json.load(open(f'{LGBM_DIR}/lgbm_clean_v1/metadata.json'))
print('=== Stage 10 reranker diagnostic (lgbm_clean_v1) ===')
print('  features:', len(meta['features']), '| best_iter:', meta.get('best_iteration'),
      '| internal val nDCG@20:', round(meta.get('best_val_ndcg20', 0), 4))

per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
sidx = labels.index('sasrec_seq')
POOL = 100
fused = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, POOL)
efpc = []
for qi, cands in enumerate(fused):
    rm = {tid: r + 1 for r, tid in enumerate(per_sub[sidx][qi])}
    efpc.append([{'sasrec_rank': rm.get(tid, 10000)} for tid in cands])
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                   model_path=f'{LGBM_DIR}/lgbm_clean_v1')
reranked = rr.rerank(queries, fused, topk=POOL, user_ids=user_ids,
                     goal_categories=goal_categories,
                     goal_specificities=goal_specificities,
                     user_profiles_raw=[None] * len(queries),
                     extra_features_per_candidate=efpc, extra_session_info=esi)

raw_ranks, new_ranks = [], []
for qi in range(len(golds)):
    g = golds[qi]
    if g in fused[qi]:
        raw_ranks.append(fused[qi].index(g) + 1)
        new_ranks.append(reranked[qi].index(g) + 1 if g in reranked[qi] else POOL + 1)
raw_ranks = np.array(raw_ranks); new_ranks = np.array(new_ranks)
nn = len(raw_ranks)
print(f'\n  golds present in pool@{POOL}: {nn} ({nn/len(golds):.1%} of all)')
print(f'  raw fused rank : median={int(np.median(raw_ranks))} mean={raw_ranks.mean():.1f} top20={(raw_ranks<=20).mean():.3f}')
print(f'  reranked rank  : median={int(np.median(new_ranks))} mean={new_ranks.mean():.1f} top20={(new_ranks<=20).mean():.3f}')
print(f'  present-gold movement: up={(new_ranks<raw_ranks).mean():.3f} down={(new_ranks>raw_ranks).mean():.3f} same={(new_ranks==raw_ranks).mean():.3f}')

print('\n  -- feature gains (lgbm_clean_v1) --')
gains = rr.booster.feature_importance(importance_type='gain')
for name, g in sorted(zip(rr.features, gains), key=lambda kv: -kv[1])[:15]:
    print(f'    {name:24s}: {g:.1f}')

print('\n-- READING --')
print('  reranked top20 >> raw top20  -> reranker helps; cap is elsewhere.')
print('  reranked top20 ~= raw top20  -> reranker near-inert; preserves fused order.')
print('  present-golds at high median -> no feature separates gold from noise')
print('     => more data / a stronger relevance feature is the lever.')


## Stage 11 - full-scale clean LGBM: does more training data help discrimination?

Stage 9 showed the reranker can't rank surfaced golds; lgbm_clean_v1 was trained on
only 2000 sessions with single-positive labels. This stage rebuilds the clean
feature set at FULL scale (15000 train / 2000 val sessions) and retrains, to test
whether the weak discrimination is a data-starvation problem.
  - feature build keeps the SASRec channel in the union (so the training candidate
    pool matches the serve pool) but DROPS the leaked columns (cfbpr_score,
    sasrec_rank_inv) -> identical clean feature set to lgbm_clean_v1.
  - compare dev nDCG@20: lgbm_clean_v1 (2k, 0.1558) vs lgbm_clean_full (15k).
COST: the feature build runs the union per train turn (~120k turns; dense re-encodes
new queries on GPU). Idempotent: skips if the full parquets already exist.

In [ ]:
# Stage 11: full-scale clean LGBM. Requires cell 1/3 (CACHE_DIR/ITEM_DB/CORPUS)
# and cell 4 (sas, queries, user_ids, ctx, golds, played, turn_numbers,
# goal_categories, goal_specificities).
import os, math
import pandas as pd
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
N_TRAIN, N_VAL = 15000, 2000
TRAIN_FULL = f'{LGBM_DIR}/lgbm_train_full_sasrec.parquet'
VAL_FULL   = f'{LGBM_DIR}/lgbm_val_full_sasrec.parquet'

# 1) Build full-scale features WITH the sasrec channel (matches serve pool).
if not os.path.exists(TRAIN_FULL):
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions {N_TRAIN} --topk 100 --seed 42 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {TRAIN_FULL}
if not os.path.exists(VAL_FULL):
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions {N_VAL} --topk 100 --seed 7 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {VAL_FULL}

# 2) Drop the leaked model-derived columns -> clean full feature set.
TRAIN_CL = f'{LGBM_DIR}/lgbm_train_full_clean.parquet'
VAL_CL   = f'{LGBM_DIR}/lgbm_val_full_clean.parquet'
LEAKED = ['cfbpr_score', 'sasrec_rank_inv']
for s, d in [(TRAIN_FULL, TRAIN_CL), (VAL_FULL, VAL_CL)]:
    df = pd.read_parquet(s)
    df.drop(columns=[c for c in LEAKED if c in df.columns]).to_parquet(d, index=False)
print('[lgbm] built full clean parquets (dropped', LEAKED, ')')

# 3) Train lgbm_clean_full.
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRAIN_CL} --val-features {VAL_CL} \
    --output-dir {LGBM_DIR}/lgbm_clean_full

# 4) Compare on dev nDCG@20 vs lgbm_clean_v1 (2k).
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
fused100 = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 100)
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

print('\n=== Stage 11 full-scale clean LGBM, DEV nDCG@20 (n=%d) ===' % len(golds))
print('  recall-only baseline:', round(ndcg20([r[:20] for r in fused100]), 4))
for name, sub in [('lgbm_clean_v1 (2k sessions)', 'lgbm_clean_v1'),
                  ('lgbm_clean_full (%dk sessions)' % (N_TRAIN // 1000), 'lgbm_clean_full')]:
    rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                       model_path=f'{LGBM_DIR}/{sub}')
    reranked = rr.rerank(queries, fused100, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories,
                         goal_specificities=goal_specificities,
                         user_profiles_raw=[None] * len(queries),
                         extra_session_info=esi)
    print('  %-32s: %.4f' % (name, ndcg20(reranked)))
print('\n  full > 2k -> data starvation was real; scale further / ship full.')
print('  full ~= 2k -> not a data problem; need better features, not more rows.')


## Stage 12 - query-signal A/B: does adding taste/intent crack the new-artist wall?

Schema audit found the retrieval query (cell 4) uses conversation turns ONLY -- it
withholds listener_goal (intent text) and preferred_musical_culture (taste). The
50% recall wall is 98.8% new-artist; session channels can't reach those, so the
ONLY path is content matching INTENT -- exactly what's withheld. This A/Bs the
query on union recall@100:
  A turns-only (reproduce 0.4989 baseline)
  B + listener_goal text
  C + listener_goal + preferred_musical_culture (+ preferred_language)
and reports new-artist rescues (golds A missed that B/C surface). Free on retrieval
(dense re-encodes the new query text once). If recall rises, this is a real recall
lever -> wire into _wrrf_union_v1 query construction + build_lgbm_features (parity).
Caveat: dense may already infer intent from conversation text; measure before believing.

In [ ]:
# Stage 12: query-signal A/B. Requires cell 1/3 (ITEM_DB/CORPUS/CACHE_DIR) and
# cell 4 (queries, user_ids, played, golds, item_db, build_user_dialog). Rebuilds
# the dev set from scratch so we can attach profile/goal to each query variant.
import numpy as np, pandas as pd
from datasets import load_dataset
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.sasrec_model import build_user_dialog

dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')

q_turns, q_goal, q_full = [], [], []      # three query variants, aligned
g2, uids, played2, udlg = [], [], [], []
for sess in dev:
    df = pd.DataFrame(sess['conversations'])
    up = sess.get('user_profile') or {}
    cg = sess.get('conversation_goal') or {}
    goal_txt = (cg.get('listener_goal') or '').strip()
    cult = (up.get('preferred_musical_culture') or '').strip()
    lang = (up.get('preferred_language') or '').strip()
    for _, music in df[df['role'] == 'music'].iterrows():
        tn = int(music['turn_number'])
        prior = df[(df['turn_number'] < tn) |
                   ((df['turn_number'] == tn) & (df['role'] == 'user'))]
        lines = []
        for _, t in prior.iterrows():
            role = 'assistant' if t['role'] == 'music' else t['role']
            content = item_db.id_to_metadata(t['content']) if t['role'] == 'music' else t['content']
            lines.append(f'{role}: {content}')
        base = chr(10).join(lines)
        q_turns.append(base)
        q_goal.append(base + (f'{chr(10)}goal: {goal_txt}' if goal_txt else ''))
        extra = (f'{chr(10)}goal: {goal_txt}' if goal_txt else '')
        if cult: extra += f'{chr(10)}taste: {cult}'
        if lang: extra += f'{chr(10)}language: {lang}'
        q_full.append(base + extra)
        g2.append(music['content']); uids.append(sess.get('user_id'))
        played2.append(list(df[(df['role'] == 'music') & (df['turn_number'] < tn)]['content']))
        udlg.append(build_user_dialog(prior.to_dict('records')))
ctx2 = [{'history_tids': p, 'user_dialog': d} for p, d in zip(played2, udlg)]
print('[stage12] built', len(q_turns), 'turns x 3 query variants')

# union+SASRec, instruct dense (current best retrieval).
uni = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                            extra_config={'use_sasrec': True, 'w_sasrec': 1.0})

def recall_at(cands, k):
    return float(np.mean([1.0 if g in c[:k] else 0.0 for c, g in zip(cands, g2)]))

def artist_of(tid):
    m = item_db.metadata_dict.get(tid, {}); a = m.get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

print('\n=== Stage 12 query-signal A/B (union+SASRec recall@100, dev n=%d) ===' % len(g2))
res = {}
for name, qs in [('A turns-only', q_turns), ('B +goal', q_goal), ('C +goal+taste', q_full)]:
    cand = uni.batch_text_to_item_retrieval(qs, topk=100, user_ids=uids, batch_context=ctx2)
    res[name] = cand
    print('  %-16s recall@20=%.4f  recall@100=%.4f' % (name, recall_at(cand, 20), recall_at(cand, 100)))

# new-artist rescues: golds A missed @100 that C now surfaces
miss_A = [i for i in range(len(g2)) if g2[i] not in set(res['A turns-only'][i][:100])]
resc = [i for i in miss_A if g2[i] in set(res['C +goal+taste'][i][:100])]
resc_new = sum(1 for i in resc
               if not (artist_of(g2[i]) is not None and artist_of(g2[i]) in {artist_of(t) for t in played2[i]}))
print('\n  A-missed golds @100: %d' % len(miss_A))
print('  rescued by C       : %d' % len(resc))
if resc:
    print('    of which new-artist: %d (%.1f%%)' % (resc_new, 100.0 * resc_new / len(resc)))
print('\n-- READING --')
print('  C/B recall > A -> withheld taste/intent IS a recall lever; wire into the query.')
print('  C/B ~= A       -> dense already captures intent from turns; query text not the gap.')


## Stage 13 - lyrics channel A/B: does a 2nd content view crack the new-artist wall?

A1 (roadmap 2026-05-30): add dense_lyrics_qwen3_instruct (precomputed lyrics-qwen3
embeddings) as a union channel. Different content view than metadata-dense, so it
can surface new-artist golds the metadata/lexical/session channels miss (the 50%
wall, 98.8% new-artist). Opt-in via use_lyrics. This A/Bs union+SASRec recall@100
WITHOUT vs WITH lyrics (sweeping its weight), and reports new-artist rescues.
Uses the A2-fixed query (cell 4, incl. listener_goal). Free-ish: dense re-encodes
queries once for the lyrics column; track embeddings precomputed.

In [ ]:
# Stage 13: lyrics channel A/B. Requires cell 4 (queries, user_ids, ctx, golds,
# played, item_db, recall_at). Baseline = union+SASRec (the shipped config 194).
from mcrs.retrieval_modules import load_retrieval_module

def artist_of(tid):
    m = item_db.metadata_dict.get(tid, {}); a = m.get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

base_cfg = {'use_sasrec': True, 'w_sasrec': 1.0}
runs = [('A union+SASRec (no lyrics)', dict(base_cfg)),
        ('B + lyrics w=0.4',          dict(base_cfg, use_lyrics=True, w_lyrics=0.4)),
        ('C + lyrics w=0.7',          dict(base_cfg, use_lyrics=True, w_lyrics=0.7))]

print('=== Stage 13 lyrics channel A/B (union+SASRec recall@100, dev n=%d) ===' % len(golds))
cand_by = {}
for name, cfg in runs:
    uni = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, extra_config=cfg)
    cand = uni.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
    cand_by[name] = cand
    print('  %-28s recall@20=%.4f  recall@100=%.4f' % (name, recall_at(cand, 20), recall_at(cand, 100)))

# new-artist rescues: golds A missed @100 that the best lyrics run now surfaces
A = cand_by['A union+SASRec (no lyrics)']
best = max([k for k in cand_by if k != 'A union+SASRec (no lyrics)'],
           key=lambda k: recall_at(cand_by[k], 100))
B = cand_by[best]
miss_A = [i for i in range(len(golds)) if golds[i] not in set(A[i][:100])]
resc = [i for i in miss_A if golds[i] in set(B[i][:100])]
resc_new = sum(1 for i in resc
               if not (artist_of(golds[i]) is not None and artist_of(golds[i]) in {artist_of(t) for t in played[i]}))
print('\n  vs %s:' % best)
print('  A-missed golds @100: %d  | rescued: %d' % (len(miss_A), len(resc)))
if resc:
    print('    of which new-artist: %d (%.1f%%)' % (resc_new, 100.0 * resc_new / len(resc)))
print('\n-- READING --')
print('  lyrics recall > A + new-artist rescues -> add use_lyrics to config 194/195, ship.')
print('  lyrics ~= A -> lyrics redundant with metadata/sasrec; drop it.')


## Stage 14 - train-on-300: does training the reranker on deeper candidates help?

Stage 9 widened the SERVE pool (100->500) with a reranker TRAINED on 100 -> recall
+0.07 but nDCG flat (reranker couldn't rank deep golds it never trained on). The
untested variant: TRAIN the reranker on a 300-candidate pool (deeper negatives +
deeper golds), then serve at 300. Cheaper than OOF (NO SASRec retrains; reuses
sasrec_v1). Compares dev nDCG@20:
  - clean_full (trained@100, served@100) = 0.1623  (shipped best)
  - clean_full (trained@100, served@300)            (the Stage 9 negative, re-confirmed)
  - clean_300  (trained@300, served@300)            (this experiment)
If clean_300@300 > 0.1623, training-on-deeper-pool converts latent recall -> ship it.

In [ ]:
# Stage 14: train-on-300. Requires cell 1/3 (CACHE_DIR/ITEM_DB/CORPUS) and cell 4
# (sas, queries, user_ids, ctx, played, golds, turn_numbers, goal_categories,
# goal_specificities). Idempotent: skips finished parquets.
import os, math
import pandas as pd
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
TR300 = f'{LGBM_DIR}/lgbm_train_full300_sasrec.parquet'
VA300 = f'{LGBM_DIR}/lgbm_val_full300_sasrec.parquet'

# 1) Build features at topk=300 (same 15k/2k scale as clean_full, with sasrec channel).
if not os.path.exists(TR300):
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 15000 --topk 300 --seed 42 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {TR300}
if not os.path.exists(VA300):
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 2000 --topk 300 --seed 7 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {VA300}

# 2) Drop leaked columns -> clean feature set (same as clean_full).
TRC = f'{LGBM_DIR}/lgbm_train_clean300.parquet'
VAC = f'{LGBM_DIR}/lgbm_val_clean300.parquet'
LEAKED = ['cfbpr_score', 'sasrec_rank_inv']
for s, d in [(TR300, TRC), (VA300, VAC)]:
    df = pd.read_parquet(s)
    df.drop(columns=[c for c in LEAKED if c in df.columns]).to_parquet(d, index=False)
print('[stage14] built clean-300 parquets (dropped', LEAKED, ')')

# 3) Train lgbm_clean_300.
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRC} --val-features {VAC} \
    --output-dir {LGBM_DIR}/lgbm_clean_300

# 4) Eval: 3 conditions. Fuse once at 300; slice to 100 for the trained@100 model.
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
fused300 = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 300)
fused100 = [f[:100] for f in fused300]
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

def rr_ndcg(sub, pool):
    mp = f'{LGBM_DIR}/{sub}'
    if not os.path.exists(f'{mp}/booster.txt'):
        return None
    rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, model_path=mp)
    reranked = rr.rerank(queries, pool, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories,
                         goal_specificities=goal_specificities,
                         user_profiles_raw=[None] * len(queries),
                         extra_session_info=esi)
    return ndcg20(reranked)

print('\n=== Stage 14 train-on-300 DEV nDCG@20 (n=%d) ===' % len(golds))
print('  recall-only @20 (fused100)        :', round(ndcg20([r[:20] for r in fused100]), 4))
a = rr_ndcg('lgbm_clean_full', fused100)
b = rr_ndcg('lgbm_clean_full', fused300)
c = rr_ndcg('lgbm_clean_300',  fused300)
print('  clean_full  trained@100 served@100:', round(a, 4) if a else 'NA', '  <- shipped 0.1623')
print('  clean_full  trained@100 served@300:', round(b, 4) if b else 'NA', '  <- Stage 9 negative')
print('  clean_300   trained@300 served@300:', round(c, 4) if c else 'NA', '  <- this experiment')
print('\n-- READING --')
if a and c:
    print('  clean_300 %+.4f vs shipped clean_full' % (c - a))
    print('  > 0 -> training on a deeper pool converts latent recall; rebuild config with topk 300.')
    print('  <=0 -> deeper-pool training does not help; reranker is feature-limited, not pool-limited.')


## Stage 15 - RRF weight + k sweep (FREE, no retrieval): tune the union mix

The union weights were hand-set and never swept: bm25=1.0, dense=0.7, same_artist=1.0,
sasrec=1.0, k=60. But sasrec is the strongest channel (0.4339) yet weighted = bm25
(0.3957), and the weak instruct-dense (0.179 isolated) gets 0.7 (likely RRF noise).
fuse_per_sub is a PURE function: we pull per-sub rankings ONCE (batch_per_sub_rankings)
then grid-search weights + k in pure Python over the cached lists — zero GPU, minutes.
Reports recall@100 (and @20) per combo; baseline = current shipped weights. RRF score
scales linearly in weight, so we fix bm25=1.0 and sweep the others' RATIO + k.

In [ ]:
# Stage 15: RRF weight + k sweep. Requires cell 4 (sas, queries, user_ids, ctx, golds).
# Pure re-fusion over cached per-sub rankings — no retriever re-run.
from mcrs.retrieval_modules.rrf import RRF_MODEL

# 1) Pull per-channel rankings ONCE (the only expensive step; reuses dense cache).
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
print('[stage15] channels:', labels)

def recall_at(cands, k):
    hit = 0
    for c, g in zip(cands, golds):
        if g in c[:k]:
            hit += 1
    return hit / len(golds)

def fuse(weight_by_label, k, topk=100):
    # align weights to per_sub order via labels
    w = [float(weight_by_label.get(lbl, 1.0)) for lbl in labels]
    return RRF_MODEL.fuse_per_sub(per_sub, w, k, topk)

# 2) Baseline = current shipped weights (bm25=1.0, dense=0.7, artist=1.0, sasrec=1.0, k=60).
BM, DE, AR, SA = 'bm25', 'dense_metadata_qwen3_instruct', 'same_artist', 'sasrec_seq'
base_w = {BM: 1.0, DE: 0.7, AR: 1.0, SA: 1.0}
base = fuse(base_w, 60)
base_r100 = recall_at(base, 100)
print('[stage15] BASELINE (shipped) recall@100=%.4f recall@20=%.4f' % (base_r100, recall_at(base, 20)))

# 3) Grid: fix bm25=1.0 (RRF is scale-invariant), sweep the rest + k.
grid_qwen   = [0.3, 0.5, 0.7, 1.0]
grid_artist = [0.7, 1.0, 1.5]
grid_sasrec = [1.0, 1.5, 2.0, 2.5]
grid_k      = [20, 40, 60]

results = []
for wq in grid_qwen:
    for wa in grid_artist:
        for ws in grid_sasrec:
            for k in grid_k:
                w = {BM: 1.0, DE: wq, AR: wa, SA: ws}
                fused = fuse(w, k)
                results.append((recall_at(fused, 100), recall_at(fused, 20), wq, wa, ws, k))

results.sort(key=lambda r: -r[0])
print('\n[stage15] top 10 combos by recall@100 (qwen/artist/sasrec/k):')
for r100, r20, wq, wa, ws, k in results[:10]:
    tag = '  *BEATS baseline' if r100 > base_r100 else ''
    print('  r@100=%.4f r@20=%.4f | qwen=%.1f artist=%.1f sasrec=%.1f k=%d%s'
          % (r100, r20, wq, wa, ws, k, tag))

best = results[0]
print('\n[stage15] BEST recall@100=%.4f vs baseline %.4f (delta %+.4f)'
      % (best[0], base_r100, best[0] - base_r100))
print('  winning weights: w_qwen=%.1f w_artist=%.1f w_sasrec=%.1f k=%d (bm25=1.0)'
      % (best[2], best[3], best[4], best[5]))
print('\n-- READING --')
print('  delta > ~0.005 -> wire winning weights into config extra_config (w_qwen/w_artist/w_sasrec)')
print('     NOTE: also rebuild LGBM features with the same weights for train/serve parity,')
print('     and the rerank fused order changes -> re-check nDCG (Stage 3/11) before shipping.')
print('  delta ~0 -> current hand-set weights were already near-optimal; move on.')


## Stage 16 - related-artist co-occurrence PROBE (no GPU, no new code): is Lever 3 worth building?

The ~50% recall wall is 98.8% NEW-ARTIST; same_artist can't reach them. Lever 3 =
a related-artist channel (cross-session artist co-occurrence). Before building a
retriever, this probe measures the UPPER BOUND: build an artist->co-occurring-artist
graph from the 15k TRAIN sessions, then for each dev gold the union MISSED @100,
check whether the gold's artist is reachable by expanding the session's seen artists
to their top-K co-occurring artists. The reachable fraction = the max a related-artist
channel could rescue. Pure counting, minutes, no model code. Decides whether to build.

In [ ]:
# Stage 16: related-artist co-occurrence probe. Requires cell 4 (cs, played, golds,
# item_db) + the train split. No GPU, no new retriever — pure counting.
from collections import Counter, defaultdict
from datasets import load_dataset
import pandas as pd

md_dict = item_db.metadata_dict
def artist_of(tid):
    a = md_dict.get(tid, {}).get('artist_name')
    if isinstance(a, list):
        a = a[0] if a else ''
    return str(a or '').strip().lower()

# 1) Build artist co-occurrence from TRAIN sessions (artists sharing a session).
print('[stage16] building artist co-occurrence from train...')
tr = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='train')
cooc = defaultdict(Counter)
for sess in tr:
    df = pd.DataFrame(sess['conversations'])
    arts = []
    for tid in df[df['role'] == 'music']['content']:
        a = artist_of(tid)
        if a:
            arts.append(a)
    uniq = list(dict.fromkeys(arts))  # unique, order-preserving
    for a in uniq:
        for b in uniq:
            if a != b:
                cooc[a][b] += 1
print('[stage16] artists with co-occurrence:', len(cooc))

# 2) For each dev turn the union MISSED @100, is the gold artist reachable by
#    expanding the session's seen artists to top-K co-occurring artists?
def reachable_at(K):
    n_miss = 0; n_new = 0; n_reach = 0
    for i in range(len(golds)):
        if golds[i] in set(cs[i][:100]):
            continue  # union already got it
        n_miss += 1
        g_art = artist_of(golds[i])
        seen = {artist_of(t) for t in played[i]}
        seen.discard('')
        is_new = g_art != '' and g_art not in seen
        if not is_new:
            continue
        n_new += 1
        # expand: rank all artists co-occurring with any seen artist, by summed count
        exp = Counter()
        for s in seen:
            for b, c in cooc.get(s, {}).items():
                if b not in seen:
                    exp[b] += c
        topK = {a for a, _ in exp.most_common(K)}
        if g_art in topK:
            n_reach += 1
    return n_miss, n_new, n_reach

print('\n=== Stage 16 related-artist rescue ceiling (union-missed dev golds) ===')
m, nw, _ = reachable_at(10)
print('  union-missed @100        : %d' % m)
print('  of which new-artist      : %d (%.1f%%)' % (nw, 100.0 * nw / max(1, m)))
print('  reachable via co-occurrence expansion (of the new-artist misses):')
for K in (10, 50, 100, 200, 500):
    _, nw2, nr = reachable_at(K)
    print('    top-%-4d co-occ artists : %d (%.1f%% of new-artist misses, %.1f%% of all misses)'
          % (K, nr, 100.0 * nr / max(1, nw2), 100.0 * nr / max(1, m)))

print('\n-- READING --')
print('  high reachable %% (e.g. >15-20%% of misses at K<=200) -> related-artist channel is')
print('     worth building (Lever 3): co-occurrence genuinely reaches the wall artists.')
print('  low reachable %% -> wall artists are cold (no train co-occurrence); the channel')
print('     would not help -> skip Lever 3, the wall needs content/intent not artist-CF.')


## Stage 17 - Lever 2: n_channels_hit reranker feature (cross-channel agreement)

Stage 10 found the reranker feature-limited (no strong relevance signal). Lever 2
adds n_channels_hit = how many union channels surfaced each candidate (golds tend
to be multiply-surfaced) — a free cross-channel-agreement relevance feature, now
plumbed train+serve (WRRFRunner.run + build_sasrec_extra_features, guarded for
parity). This rebuilds full features (which now include the column), trains
lgbm_clean_full_nch (clean feature set + n_channels_hit), and compares dev nDCG@20
vs clean_full (0.1623). COST: one 15k feature rebuild (~40min, the column needs
fresh parquets) + CPU train. Idempotent.

In [ ]:
# Stage 17: n_channels_hit reranker feature. Requires cell 1/3 + cell 4 (sas,
# queries, user_ids, ctx, golds, played, turn_numbers, goal_categories,
# goal_specificities). Idempotent (skips finished parquets).
import os, math
import pandas as pd
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
# Fresh full-scale parquets that include n_channels_hit (the pre-Lever-2 parquets
# lack the column). Distinct filenames so we don't clobber lgbm_*_full_sasrec.
TR = f'{LGBM_DIR}/lgbm_train_full_nch.parquet'
VA = f'{LGBM_DIR}/lgbm_val_full_nch.parquet'
if not os.path.exists(TR):
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 15000 --topk 100 --seed 42 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {TR}
if not os.path.exists(VA):
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 2000 --topk 100 --seed 7 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {VA}

# Clean feature set (drop leaked cols) but KEEP n_channels_hit.
TRC = f'{LGBM_DIR}/lgbm_train_clean_nch.parquet'
VAC = f'{LGBM_DIR}/lgbm_val_clean_nch.parquet'
LEAKED = ['cfbpr_score', 'sasrec_rank_inv']
for s, d in [(TR, TRC), (VA, VAC)]:
    df = pd.read_parquet(s)
    assert 'n_channels_hit' in df.columns, 'rebuild missing n_channels_hit — pull latest code'
    df.drop(columns=[c for c in LEAKED if c in df.columns]).to_parquet(d, index=False)
print('[stage17] clean+nch parquets ready; n_channels_hit present')

!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRC} --val-features {VAC} \
    --output-dir {LGBM_DIR}/lgbm_clean_full_nch

# Eval vs clean_full. Feed efpc with BOTH sasrec_rank (ignored by clean models)
# and n_channels_hit, computed from per_sub exactly like build_sasrec_extra_features.
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
fused100 = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 100)
sidx = labels.index('sasrec_seq')
efpc = []
for qi, cands in enumerate(fused100):
    rm = {tid: r + 1 for r, tid in enumerate(per_sub[sidx][qi])}
    hit = {}
    for s in range(len(per_sub)):
        for t in per_sub[s][qi]:
            hit[t] = hit.get(t, 0) + 1
    efpc.append([{'sasrec_rank': rm.get(tid, 10000),
                  'n_channels_hit': hit.get(tid, 1)} for tid in cands])
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

print('\n=== Stage 17 n_channels_hit DEV nDCG@20 (n=%d) ===' % len(golds))
print('  recall-only:', round(ndcg20([r[:20] for r in fused100]), 4))
for name, sub in [('clean_full      (no nch)', 'lgbm_clean_full'),
                  ('clean_full_nch  (+nch)  ', 'lgbm_clean_full_nch')]:
    mp = f'{LGBM_DIR}/{sub}'
    if not os.path.exists(f'{mp}/booster.txt'):
        print('  ' + name + ' : (missing)'); continue
    rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, model_path=mp)
    reranked = rr.rerank(queries, fused100, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories,
                         goal_specificities=goal_specificities,
                         user_profiles_raw=[None] * len(queries),
                         extra_features_per_candidate=efpc, extra_session_info=esi)
    print('  ' + name + ' :', round(ndcg20(reranked), 4))
print('\n-- READING: clean_full_nch > 0.1623 -> n_channels_hit helps; ship it (config 195). --')


## Stage 18 - undertrained reranker: more boosting rounds (n_estimators)

Stage 11/14 LGBM logs show "Did not meet early stopping. Best iteration is [1000]"
with val nDCG STILL RISING monotonically at the cap — the reranker is UNDERTRAINED,
truncated by n_estimators=1000, not converged. This retrains clean_full on the SAME
parquets (no rebuild) at higher caps and compares dev nDCG@20 vs 0.1637.
CAVEAT: the rising curve is INTERNAL val (optimistic, conditional-on-present);
more rounds may just overfit. Must judge on DEV, not internal val. Cheap (CPU,
minutes — just more boosting on existing features).

In [ ]:
# Stage 18: more boosting rounds. Requires cell 1/3 + cell 4. Reuses the Stage 11
# clean full parquets (no feature rebuild). Idempotent per output dir.
import os, math
import pandas as pd
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
TRC = f'{LGBM_DIR}/lgbm_train_full_clean.parquet'
VAC = f'{LGBM_DIR}/lgbm_val_full_clean.parquet'
assert os.path.exists(TRC), 'Run Stage 11 first (lgbm_train_full_clean.parquet missing)'

# Train at increasing round caps. lr stays 0.05 (hardcoded in train_lgbm_ranker);
# early-stopping=50 still active, so a cap that converges will stop on its own.
for n_est in (2000, 4000):
    out = f'{LGBM_DIR}/lgbm_clean_full_n{n_est}'
    if os.path.exists(f'{out}/booster.txt'):
        print(f'[stage18] {out} present, skip'); continue
    !cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
        --train-features {TRC} --val-features {VAC} \
        --n-estimators {n_est} --early-stopping 100 \
        --output-dir {out}

# Eval on dev vs shipped clean_full (n=1000).
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
fused100 = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 100)
sidx = labels.index('sasrec_seq')
efpc = []
for qi, cands in enumerate(fused100):
    rm = {tid: r + 1 for r, tid in enumerate(per_sub[sidx][qi])}
    efpc.append([{'sasrec_rank': rm.get(tid, 10000)} for tid in cands])
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

print('\n=== Stage 18 more-rounds DEV nDCG@20 (n=%d) ===' % len(golds))
print('  recall-only:', round(ndcg20([r[:20] for r in fused100]), 4))
for name, sub in [('clean_full  n=1000 (shipped)', 'lgbm_clean_full'),
                  ('clean_full  n=2000', 'lgbm_clean_full_n2000'),
                  ('clean_full  n=4000', 'lgbm_clean_full_n4000')]:
    mp = f'{LGBM_DIR}/{sub}'
    if not os.path.exists(f'{mp}/booster.txt'):
        print('  ' + name + ' : (missing)'); continue
    rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, model_path=mp)
    reranked = rr.rerank(queries, fused100, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories,
                         goal_specificities=goal_specificities,
                         user_profiles_raw=[None] * len(queries),
                         extra_features_per_candidate=efpc, extra_session_info=esi)
    print('  ' + name + ' :', round(ndcg20(reranked), 4))
print('\n-- READING --')
print('  dev nDCG rises with rounds -> was undertrained; ship the best n_estimators.')
print('  dev nDCG flat/falls -> internal-val rise was overfitting; keep n=1000.')


## Stage 19 - Lever 4: cf-bpr union channel A/B (orthogonal user-taste signal)

cf-bpr (user x item affinity) is built + factory-wired but NOT in the union — only
an LGBM feature. As a recall channel it's ORTHOGONAL to session (same_artist/sasrec)
and content (bm25/dense): for warm users it can surface popular tracks by NEW artists
their latent taste likes. Opt-in use_cfbpr (default w=0.25; ~43% warm, cold -> empty
list, RRF falls back). A/Bs union+SASRec recall@100 without vs with cf-bpr, with a
warm/cold split (the lever only helps warm users; must not regress cold). Free-ish:
cf-bpr is numpy-only, no GPU encode.

In [ ]:
# Stage 19: cf-bpr union channel A/B. Requires cell 4 (queries, user_ids, ctx,
# golds, played, item_db, recall_at, user_dialogs).
from mcrs.retrieval_modules import load_retrieval_module

def artist_of(tid):
    a = item_db.metadata_dict.get(tid, {}).get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

base_cfg = {'use_sasrec': True, 'w_sasrec': 1.0}
runs = [('A union+SASRec (no cfbpr)', dict(base_cfg)),
        ('B + cfbpr w=0.25',         dict(base_cfg, use_cfbpr=True, w_cfbpr=0.25)),
        ('C + cfbpr w=0.5',          dict(base_cfg, use_cfbpr=True, w_cfbpr=0.5))]

# warm = user has a cf-bpr embedding. Load the cf_bpr retriever once to get the set.
cfr = load_retrieval_module('cf_bpr', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, extra_config={})
warm = set(cfr.user_embs.keys())
warm_mask = [uid in warm for uid in user_ids]
n_warm = sum(warm_mask)
print('[stage19] warm users: %d/%d (%.1f%%)' % (n_warm, len(user_ids), 100.0*n_warm/len(user_ids)))

def recall_split(cands, k):
    def r(mask):
        idx = [i for i in range(len(golds)) if mask[i]]
        if not idx: return float('nan')
        return sum(1.0 for i in idx if golds[i] in set(cands[i][:k])) / len(idx)
    return recall_at(cands, k), r(warm_mask), r([not m for m in warm_mask])

print('\n=== Stage 19 cf-bpr union A/B (recall@100, dev n=%d) ===' % len(golds))
cand_by = {}
for name, cfg in runs:
    uni = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, extra_config=cfg)
    cand = uni.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
    cand_by[name] = cand
    overall, warm_r, cold_r = recall_split(cand, 100)
    print('  %-26s overall=%.4f  warm=%.4f  cold=%.4f' % (name, overall, warm_r, cold_r))

print('\n-- READING --')
print('  warm recall rises AND cold recall not down -> cf-bpr is an orthogonal win; ship use_cfbpr.')
print('  warm ~flat or cold regresses -> cf-bpr adds RRF noise; drop it.')


## Stage 20 - does the Stage 15 weight win convert to nDCG?

Stage 15 found w_artist=1.5, w_sasrec=1.5 (bm25=1.0, qwen=0.7, k=60) lifts
recall@100 0.5061 -> 0.5154. But recall up != nDCG up (Stages 8/9/14). This
reranks BOTH the baseline-weight and new-weight fused pools with the SAME shipped
clean_full reranker and compares dev nDCG@20. APPROXIMATE: clean_full was trained
on old-weight features, so this under-credits the new weights slightly. If nDCG
holds/rises even approximately -> worth a full feature rebuild at the new weights.

In [ ]:
# Stage 20: reweighted-union nDCG check. Requires cell 4 (sas, queries, user_ids,
# ctx, golds, played, turn_numbers, goal_categories, goal_specificities).
import math
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
def wvec(by_label):
    return [float(by_label.get(l, 1.0)) for l in labels]
BM, DE, AR, SA = 'bm25', 'dense_metadata_qwen3_instruct', 'same_artist', 'sasrec_seq'
base_w = {BM:1.0, DE:0.7, AR:1.0, SA:1.0}
new_w  = {BM:1.0, DE:0.7, AR:1.5, SA:1.5}

def recall_at(c, k): return sum(1.0 for x,g in zip(c,golds) if g in set(x[:k]))/len(golds)
def ndcg20(ranked):
    s=0.0
    for r,g in zip(ranked,golds):
        for pos,tid in enumerate(r[:20]):
            if tid==g: s+=1.0/math.log2(pos+2); break
    return s/len(golds)

rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                   model_path=f'{CACHE_DIR}/retrieval_v2/lgbm/lgbm_clean_full')
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]
sidx = labels.index('sasrec_seq')

print('=== Stage 20 reweighted-union nDCG@20 (clean_full reranker, dev n=%d) ===' % len(golds))
for name, w in [('baseline weights', base_w), ('NEW weights (a=1.5,s=1.5)', new_w)]:
    fused = RRF_MODEL.fuse_per_sub(per_sub, wvec(w), 60, 100)
    efpc = []
    for qi, cands in enumerate(fused):
        rm = {tid: r+1 for r,tid in enumerate(per_sub[sidx][qi])}
        efpc.append([{'sasrec_rank': rm.get(tid,10000)} for tid in cands])
    reranked = rr.rerank(queries, fused, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories, goal_specificities=goal_specificities,
                         user_profiles_raw=[None]*len(queries),
                         extra_features_per_candidate=efpc, extra_session_info=esi)
    print('  %-26s recall@100=%.4f  nDCG@20=%.4f' % (name, recall_at(fused,100), ndcg20(reranked)))
print('\n-- READING: NEW nDCG >= baseline -> rebuild features at new weights + ship config 195. --')
print('   (approximate: clean_full trained on old-weight features, so this under-credits NEW.)')


## Stage 21 - Lever 3: related-artist channel A/B (attacks the new-artist wall)

Stage 16 probe: ~29% of union-missed golds reachable via artist co-occurrence at
top-100 — the biggest recall ceiling of the session, and the ONLY lever aimed at
the ~96%-new-artist wall. This adds the RelatedArtistRetriever as an opt-in 5th
union channel (use_related_artist) and A/Bs union+SASRec recall@100 without vs with
it (weight sweep), with the new-artist-rescue breakdown. First run builds + caches
the co-occurrence map from train (~1-2 min, no GPU); later runs load the cache.

In [ ]:
# Stage 21: related-artist channel A/B. Requires cell 4 (queries, user_ids, ctx,
# golds, played, item_db, recall_at).
from mcrs.retrieval_modules import load_retrieval_module

def artist_of(tid):
    a = item_db.metadata_dict.get(tid, {}).get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

base_cfg = {'use_sasrec': True, 'w_sasrec': 1.0}
runs = [('A union+SASRec (no related)', dict(base_cfg)),
        ('B + related w=0.5',           dict(base_cfg, use_related_artist=True, w_related_artist=0.5)),
        ('C + related w=1.0',           dict(base_cfg, use_related_artist=True, w_related_artist=1.0)),
        ('D + related w=1.5',           dict(base_cfg, use_related_artist=True, w_related_artist=1.5))]

print('=== Stage 21 related-artist channel A/B (recall@100, dev n=%d) ===' % len(golds))
cand_by = {}
for name, cfg in runs:
    uni = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, extra_config=cfg)
    cand = uni.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
    cand_by[name] = cand
    print('  %-28s recall@20=%.4f  recall@100=%.4f' % (name, recall_at(cand, 20), recall_at(cand, 100)))

A = cand_by['A union+SASRec (no related)']
best = max([k for k in cand_by if k != 'A union+SASRec (no related)'], key=lambda k: recall_at(cand_by[k], 100))
B = cand_by[best]
miss_A = [i for i in range(len(golds)) if golds[i] not in set(A[i][:100])]
resc = [i for i in miss_A if golds[i] in set(B[i][:100])]
resc_new = sum(1 for i in resc
               if not (artist_of(golds[i]) is not None and artist_of(golds[i]) in {artist_of(t) for t in played[i]}))
print('\n  vs %s:' % best)
print('  A-missed @100: %d  | rescued: %d' % (len(miss_A), len(resc)))
if resc:
    print('    of which new-artist: %d (%.1f%%)' % (resc_new, 100.0*resc_new/len(resc)))
print('\n-- READING --')
print('  related recall > A with new-artist rescues -> the wall lever WORKS; add use_related_artist')
print('     to config 195, rebuild LGBM features at that union, re-check nDCG, ship.')
print('  related ~= A -> co-occurrence ranking too noisy at serve; tune weight / artist-topk.')
